# 03 — Bootstrap Auto-Label + Audit Loop

Objective: generate pseudo-labels with the current detector, audit them visually, and merge corrected labels for the next detector iteration.

In [ ]:
from pathlib import Path

import cv2
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:
ROOT = Path('C:/path/to/private/data')
MODEL_PATH = ROOT / 'runs' / 'stage2_fast_stable' / 'weights' / 'best.pt'
BOOTSTRAP = ROOT / 'Bootstrap_5k_v2'
IMAGES_DIR = BOOTSTRAP / 'images'
LABELS_DIR = BOOTSTRAP / 'labels'
AUDIT_DIR = ROOT / 'bootstrap_v2_audit'
AUDIT_IMAGES = AUDIT_DIR / 'images'
AUDIT_LABELS = AUDIT_DIR / 'labels'

for p in [LABELS_DIR, AUDIT_IMAGES, AUDIT_LABELS]:
    p.mkdir(parents=True, exist_ok=True)

image_files = [p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
print('Images to process:', len(image_files))

In [ ]:
# Step A: pseudo-label generation
model = YOLO(str(MODEL_PATH))

for image_path in tqdm(image_files, desc='Auto-label'):
    model.predict(
        source=str(image_path),
        conf=0.15,
        imgsz=640,
        save=False,
        save_txt=True,
        project=str(BOOTSTRAP),
        name='labels',
        exist_ok=True,
        verbose=False
    )

In [ ]:
# Step B: ensure one label file per image
created_empty = 0
for img in image_files:
    label = LABELS_DIR / f'{img.stem}.txt'
    if not label.exists():
        label.write_text('')
        created_empty += 1

print('created_empty =', created_empty)

In [ ]:
# Step C: export audit overlays
for img in tqdm(image_files, desc='Render audit'):
    frame = cv2.imread(str(img))
    if frame is None:
        continue

    h, w = frame.shape[:2]
    label_file = LABELS_DIR / f'{img.stem}.txt'
    if label_file.exists():
        for line in label_file.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            _, xc, yc, bw, bh = map(float, parts)
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    cv2.imwrite(str(AUDIT_IMAGES / img.name), frame)

print('Audit overlays exported to', AUDIT_IMAGES)

In [ ]:
# Step D: merge manual corrections
# Manual operation between C and D: place corrected txt files in AUDIT_LABELS
deleted_old, copied_manual, created_empty = 0, 0, 0

for audit_img in tqdm(AUDIT_IMAGES.iterdir(), desc='Merge corrected labels'):
    if audit_img.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:
        continue

    label_name = f'{audit_img.stem}.txt'
    dst = LABELS_DIR / label_name
    src = AUDIT_LABELS / label_name

    if dst.exists():
        dst.unlink()
        deleted_old += 1

    if src.exists():
        dst.write_text(src.read_text())
        copied_manual += 1
    else:
        dst.write_text('')
        created_empty += 1

print('deleted_old   =', deleted_old)
print('copied_manual =', copied_manual)
print('created_empty =', created_empty)

## Deliverable
- Bootstrap labels generated
- Audit visualizations exported
- Corrected labels merged for next training cycle